In [1]:
import torch.nn as nn
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import numpy as np
import csv
import os
torch.cuda.empty_cache()

class StrainDataset(Dataset):

    def __init__(self, path, mode='train', sequence_length=500, transforms=None):
        self.path = path
        self.mode = mode
        self.transforms = transforms
        self.seq_len = sequence_length
        # read data into numpy arrays
        with open(path, 'r') as fp:
            data = list(csv.reader(fp))
            data = np.array(data)
            features = data[1::30, 2:5]  
            target = data[1::30, 5:6] 
            all_time = data[1::30, 2:3]  
            resistance_f = data[1::30, 2:3]
            cycle_num = data[1::30, 3:4]

            y = target.astype(float)
            self.y = torch.tensor(y)
            X = features.astype(float)
            self.X = torch.tensor(X)
            time = all_time.astype(float)
            self.time = torch.tensor(time)
            resis_f = resistance_f.astype(float)
            self.resis_f = torch.tensor(resis_f)
            cyc_n = cycle_num.astype(float)
            self.cyc_n = torch.tensor(cyc_n)

            self.X[:, :] = \
                (self.X[:, :] - self.X[:, :].mean(dim=0, keepdim=True)) \
                / self.X[:, :].std(dim=0, keepdim=True)
        print('Finished reading the {} set of Strain Dataset ({} samples found)'.format(mode, len(self.X)))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        if index >= self.seq_len - 1:
            i_start = index - self.seq_len + 1
            x = self.X[i_start:(index + 1), :]
            time = self.time[i_start:(index + 1), :]
            resist = self.resis_f[i_start:(index + 1), :]
            cycle = self.cyc_n[i_start:(index + 1), :]
        else:
            padding = self.X[0].repeat(self.seq_len - index - 1, 1)
            x = self.X[0:(index + 1), :]
            x = torch.cat((padding, x), 0)
            padding_time = self.time[0].repeat(self.seq_len - index - 1, 1)
            time = self.time[0:(index + 1), :]
            time = torch.cat((padding_time, time), 0)
            padding_resist = self.resis_f[0].repeat(self.seq_len - index - 1, 1)
            resist = self.resis_f[0:(index + 1), :]
            resist = torch.cat((padding_resist, resist), 0)
            padding_cycle = self.cyc_n[0].repeat(self.seq_len - index - 1, 1)
            cycle = self.cyc_n[0:(index + 1), :]
            cycle = torch.cat((padding_cycle, cycle), 0)
        return x, self.y[index], time, resist, cycle

def get_device():
    return 'cuda'if torch.cuda.is_available() else 'cpu'
device = get_device()

class LSTMpred(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, bias=True,
                 batch_first=False, dropout=0, output_size=1):
        super().__init__()
        self.input_dim = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

        self.linear1 = nn.Linear(hidden_size, 128)
        self.linear2 = nn.Linear(128, 1)

        self.hidden = self.init_hidden()
    def init_hidden(self):
        return (torch.zeros(self.num_layers, 36, self.hidden_size).to(device),
			    torch.zeros(self.num_layers, 36, self.hidden_size).to(device))

    def forward(self, x):
        lstm_out, _ = self.lstm(x, self.hidden)
        pred = self.linear1(lstm_out[:, -1, :])
        pred = self.linear2(pred)
        return pred

model = LSTMpred(input_size=3, hidden_size=256, num_layers=2, batch_first=True)  
loss_funtion = nn.MSELoss()
# training
def train(model_tr_data,dv_set,model,config,device):

    n_epochs = config['n_epochs']   
    optimizer = getattr(torch.optim, config['optimizer'])(
        model.parameters(), **config['optim_hparas'])
    min_mse = 1000
    loss_record = {'train': [], 'dev': []}   
    early_stop_cnt = 0
    epoch = 0
    num_batche_tr = len(model_tr_data)
    print(num_batche_tr)
    while epoch < n_epochs:
        print(epoch)
        allbat_mse_loss = 0
        model.train() 
        for x, y, t, r, c in model_tr_data: 
            model.zero_grad() 
            x = x.float().to(device)
            pred = model(x)   
            y = y.float().to(device)
            mse_loss = loss_funtion(pred, y)   
            allbat_mse_loss += mse_loss
            mse_loss.backward()  
            optimizer.step()   
        evepoch_mse_loss = allbat_mse_loss / num_batche_tr

        print(evepoch_mse_loss)
        loss_record['train'].append(evepoch_mse_loss.detach().cpu().item())

        dev_mse = dev(dv_set, model, device)

        if dev_mse < min_mse:   
            min_mse = dev_mse
            print('Saving model (epoch = {},loss = {:.4f}'.format((epoch + 1), (min_mse)))
            torch.save(model.state_dict(), config ['save_path'])     
            early_stop_cnt = 0
        else:
            early_stop_cnt = early_stop_cnt + 1
        epoch = epoch + 1
        loss_record['dev'].append(dev_mse)
        if early_stop_cnt > config['early_stop']:
            break
    print('Finished training after {} epochs'.format(epoch))
    return min_mse, loss_record 

def dev(dv_set, model, device):
    model.eval()  
    total_loss = 0
    num_batches = len(dv_set)
    for i, y, t, r, c in dv_set:
        x = i.float().to(device)
        with torch.no_grad():
            pred = model(x)
            y = y.float().to(device)
            mse_loss = loss_funtion(pred, y)
        total_loss += mse_loss.detach().cpu().item()
    avg_loss = total_loss / num_batches  

    return avg_loss

def test(tt_set, model, device):
    model.eval()
    preds = []
    test_loss = 0.0
    num_batches = len(tt_set)
    for x, y, t, r, c in tt_set:  
        x = x.float().to(device)
        y = y.float().to(device)
        with torch.no_grad():
            pred = model(x)
            batch_loss = loss_funtion(pred, y)
            preds.append(pred.detach().cpu())  
        test_loss += batch_loss.detach().cpu().item()
    avg_loss_test = test_loss / num_batches
    preds = torch.cat(preds, dim=0).numpy()
    print('Finished testing predictions!')
    return preds, avg_loss_test  

def plot_learning_curve(loss_record, title=''):
    total_epochs = len(loss_record['train'])
    x_1 = range(total_epochs)
    x_2 = x_1[::len(loss_record['train']) // len(loss_record['dev'])]   
    figure(figsize=(6, 4))
    plt.plot(x_1, loss_record['train'], c='tab:red', label= 'train')
    plt.plot(x_2, loss_record['dev'], c='tab:cyan', label='dev')
    plt.ylim(0.0, 100.0)
    plt.xlabel('Training epochs')
    plt.ylabel('MSE loss')
    plt.title('Learning curve of {}'.format(title))
    plt.legend()    
    plt.show()
    
def plot_pred(dv_x,model,device,lim_x=60,lim_y=4000.,preds=None,targets=None, title=''):     
    if preds is None or targets is None:
        model.eval()
        preds, targets = [], []
        resist = []
        for i, y, t, r, c in dv_x:
            x = i.float().to(device)
            y = y.float().to(device)
            r = r.float().to(device)

            with torch.no_grad():
                pred = model(x)
                resist.append(r[:, -1, :].detach().cpu())
                preds.append(pred.detach().cpu())
                targets.append(y.detach().cpu())
        preds = torch.cat(preds, dim=0).numpy()
        targets = torch.cat(targets, dim=0).numpy()
        resist_values = torch.cat(resist, dim=0).numpy()

    fig, ax1 = plt.subplots()
    ax1.set_xlabel('resistance')
    ax1.set_ylabel('ground truth value', color='red')
    ax1.scatter(resist_values, targets, color='red')
    ax1.tick_params(axis='y', labelcolor='red')
    ax2 = ax1.twinx()
    ax2.set_ylabel('predicted value', color='blue')
    ax2.scatter(resist_values, preds, color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')
    plt.title('Ground Truth v.s. Prediction of {}'.format(title))
    plt.show()

def plot_pred_time(dv_x, model, device, preds=None, targets=None, title=''):  # 绘制验证集的预测值和实际值
    if preds is None or targets is None:
        model.eval()
        preds, targets = [], []
        all_time = []
        for i, y, t, r, c in dv_x:
            x = i.float().to(device)
            y = y.float().to(device)
            t = t.float().to(device)

            with torch.no_grad():
                pred = model(x)
                all_time.append(t[:, -1, :].detach().cpu())
                preds.append(pred.detach().cpu())
                targets.append(y.detach().cpu())
        preds = torch.cat(preds, dim=0).numpy()
        targets = torch.cat(targets, dim=0).numpy()
        time_values = torch.cat(all_time, dim=0).numpy()

    fig, ax1 = plt.subplots()
    ax1.set_xlabel('all time')
    ax1.set_ylabel('ground truth value', color='red')
    ax1.scatter(time_values, targets, color='red')
    ax1.tick_params(axis='y', labelcolor='red')
    # adding twin Axes
    ax2 = ax1.twinx()
    ax2.set_ylabel('predicted value', color='blue')
    ax2.scatter(time_values, preds, color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')
    plt.title('Ground Truth v.s. Prediction of {}'.format(title))
    plt.show()
    

os.makedirs('BatchDiversity_LSTM_models/BatchDiversity_LSTM_01_1589Preds3', exist_ok=True)   # The trained model will be saved to ./models/
config = { 
        'n_epochs': 100,   
        'batch_size': 36,   
        'optimizer': 'Adam', 
        'optim_hparas': {    
        'lr': 0.0001,   
        },
        'early_stop': 100,   
        'save_path': 'BatchDiversity_LSTM_models/BatchDiversity_LSTM_01_1589Preds3.pth'  
}

dataset = StrainDataset('./Training_batchdiversity_1589.csv', 'train', 30, transforms=None)  
dataset_for_test_01 = StrainDataset('./Testing_batchdiversity_10.csv', 'test', 30, transforms=None)

def train_dev_data(x_set):
    X = []
    deX = []
    for i in range(len(x_set)):
        if i % 10 != 0:
            x_data = x_set[i]
            X.append(x_data)
        else:
            devX = x_set[i]
            deX.append(devX)
    return X, deX 
training_data ,valid_data = train_dev_data(dataset)
batch_size = 36
torch.manual_seed(99)
model_tr_data = DataLoader(training_data, batch_size=batch_size, shuffle=True, drop_last=True)
model_dev_data = DataLoader(valid_data, batch_size=batch_size, shuffle=True, drop_last=True)
model_tt_dataX01 = DataLoader(dataset_for_test_01, batch_size=batch_size, shuffle=False, drop_last=True)
model = model.to(device)
# # start training
# model_loss, model_loss_record = train(model_tr_data, model_dev_data,model, config, device)
# total_params = sum(p.numel() for p in model.parameters())
# print('total params = ', total_params)
# print(model_loss)
# len_tr = model_loss_record['train']
# print(len_tr)
# len_dev = model_loss_record['dev']
# print(len(len_dev))

del model
model = LSTMpred(input_size=3, hidden_size=256, num_layers=2, batch_first=True)
model = model.to(device)
ckpt = torch.load(config['save_path'], map_location='cpu')   # load your best model
model.load_state_dict(ckpt)


def save_pred(preds, file):
    '''Save predictions to specified file'''
    print('Saving results to {}'.format(file))
    with open(file, 'w') as fp:
        writer = csv.writer(fp)
        writer.writerow(['id', 'strain'])
        for i, p in enumerate(preds):    
            writer.writerow([i, p])

preds01, test_loss01 = test(model_tt_dataX01, model, device) 
print(preds01)
print(test_loss01)

save_pred(preds01, 'BatchDiversity_LSTM_01_1589Preds10.csv')  

print('ALL Done!')

Finished reading the train set of Strain Dataset (325878 samples found)
Finished reading the test set of Strain Dataset (81620 samples found)
Finished testing predictions!
[[20.759766]
 [27.767612]
 [39.987984]
 ...
 [38.23429 ]
 [36.781242]
 [11.595348]]
2.7664716678191295
Saving results to BatchDiversity_LSTM_01_1589Preds10.csv
ALL Done!
